<a href="https://colab.research.google.com/github/Esha-Rahim/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esha-Rahim/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents a single (query, landing_page, country, device) entity aggregated over a daily window. The observation feature window spans 28 days prior to the cutoff date, and the outcome/label window measures conversions/clicks over the following 7-day period.

In [1]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
dataset_path = "FlyRank/internship-warehouse"
target_config = "fact_content_daily_performance"

# Stream the dataset instead of pulling all 39 shards into RAM
streaming_dataset = load_dataset(dataset_path, target_config, token=hf_token, streaming=True)

# Fetch the first 100,000 rows instantly into Pandas
records = []
for i, row in enumerate(streaming_dataset['train']):
    records.append(row)
    if i >= 99999:  # Change threshold as needed
        break

df = pd.DataFrame(records)
print(f"Loaded streamed sample into 'df'. Shape: {df.shape}")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loaded streamed sample into 'df'. Shape: (100000, 30)


In [3]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# 1. Check for date column safely
date_col = next((c for c in ['date', 'dt', 'created_at', 'timestamp'] if c in df.columns), None)

# 2. Slice dataset safely
df_mid = df[df['month'] == '2026-03'].copy() if 'month' in df.columns else df.copy()

if date_col:
    print("Date range:", df_mid[date_col].min(), "to", df_mid[date_col].max())
else:
    print("No explicit date column found. Available columns:", df.columns.tolist())

print("Total rows:", len(df_mid))

# 3. Verify grain uniqueness dynamically using columns that exist
grain_cols = ['query', 'landing_page', 'country', 'device', date_col]
existing_grain_cols = [c for c in grain_cols if c and c in df_mid.columns]

if existing_grain_cols:
    duplicates = df_mid.duplicated(subset=existing_grain_cols).sum()
    print(f"Duplicate rows for grain {existing_grain_cols}: {duplicates}")
else:
    print("None of the specified grain columns exist in this table.")

No explicit date column found. Available columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
Total rows: 100000
None of the specified grain columns exist in this table.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect schema and column data types
print("--- Schema Overview ---")
df.info()

print("\n--- Sample Values ---")
df.head(2).T

--- Schema Overview ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 30 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   report_date               100000 non-null  object 
 1   client_hash_id            100000 non-null  object 
 2   content_hash_id           100000 non-null  object 
 3   client_has_gsc            100000 non-null  bool   
 4   client_has_ga4            100000 non-null  bool   
 5   gsc_data_available        100000 non-null  bool   
 6   ga4_data_available        100000 non-null  bool   
 7   gsc_impressions           100000 non-null  int64  
 8   gsc_clicks                100000 non-null  int64  
 9   gsc_sum_position          99999 non-null   float64
 10  gsc_avg_position          99999 non-null   float64
 11  ga4_pageviews             100000 non-null  int64  
 12  ga4_sessions              100000 non-null  int64  
 13  ga4_users            

,0,1
report_date,2025-01-27,2025-01-27
client_hash_id,client_9958f0a7ae1df715,client_9958f0a7ae1df715
content_hash_id,content_3b70a18ea133b2bb,content_fe8e8155ce1d47a2
client_has_gsc,True,True
client_has_ga4,True,True
gsc_data_available,True,True
ga4_data_available,False,False
gsc_impressions,30,5
gsc_clicks,0,0
gsc_sum_position,115.0,358.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# 1. Grain Uniqueness Check
grain = ['client_hash_id', 'content_hash_id', 'report_date']
existing_grain = [c for c in grain if c in df.columns]
dup_count = df.duplicated(subset=existing_grain).sum()
print(f"1. Grain Uniqueness Check ({existing_grain}): {dup_count} duplicate rows found.")

# 2. Total Row & Column Counts
print(f"2. Total Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")

# 3. Missing Value Audit (Top 10 columns with highest nulls)
null_counts = df.isnull().sum()
null_percent = (null_counts / len(df)) * 100
null_summary = pd.DataFrame({'null_count': null_counts, 'null_percent': null_percent})
print("\n3. Top Missing Value Summary:")
print(null_summary.sort_values(by='null_count', ascending=False).head(10))

# 4. Time Window / Date Range Verification
if 'report_date' in df.columns:
    df['report_date'] = pd.to_datetime(df['report_date'])
    print(f"\n4. Date Range: Min = {df['report_date'].min().date()} | Max = {df['report_date'].max().date()}")

1. Grain Uniqueness Check (['client_hash_id', 'content_hash_id', 'report_date']): 0 duplicate rows found.
2. Total Dataset Shape: 100000 rows, 30 columns

3. Top Missing Value Summary:
                    null_count  null_percent
gsc_avg_position             1         0.001
gsc_sum_position             1         0.001
content_hash_id              0         0.000
client_has_gsc               0         0.000
report_date                  0         0.000
client_hash_id               0         0.000
gsc_data_available           0         0.000
client_has_ga4               0         0.000
gsc_impressions              0         0.000
ga4_data_available           0         0.000

4. Date Range: Min = 2025-01-27 | Max = 2025-03-21


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# 1. Check for unbalanced history across clients
if 'client_hash_id' in df.columns and 'report_date' in df.columns:
    df['report_date'] = pd.to_datetime(df['report_date'])
    client_history = df.groupby('client_hash_id')['report_date'].agg(['min', 'max', 'count'])
    print("--- 1. Client History Spans (First 5) ---")
    print(client_history.head(5))

# 2. Check GSC integration availability / flag bias
if 'client_has_gsc' in df.columns:
    gsc_counts = df['client_has_gsc'].value_counts(dropna=False)
    print("\n--- 2. GSC Coverage Distribution ---")
    print(gsc_counts)

# 3. Check window gaps / date continuity
if 'report_date' in df.columns:
    unique_dates = df['report_date'].drop_duplicates().sort_values()
    date_gaps = unique_dates.diff().dt.days
    max_gap = date_gaps.max()
    print(f"\n--- 3. Max Date Gap in Loaded Slice: {max_gap} day(s) ---")

--- 1. Client History Spans (First 5) ---
                               min        max  count
client_hash_id                                      
client_73cda7b4e4f265ea 2025-02-11 2025-03-21  53897
client_9958f0a7ae1df715 2025-01-27 2025-03-21  42753
client_fef1a8f436438636 2025-03-15 2025-03-21    815
client_ff644d8251367cbb 2025-01-27 2025-03-20   2535

--- 2. GSC Coverage Distribution ---
client_has_gsc
True    100000
Name: count, dtype: int64

--- 3. Max Date Gap in Loaded Slice: 12.0 day(s) ---


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.